# f_new6_to_4chip -- External Generalisation Result Table

Loads `08_cross_dataset_predict_new_chip.py`'s saved `predictions_final_6_new_*.joblib`
files (produced by `slurm_jobs/f_new6_to_4chip.sh`) and formats them into a LaTeX table:
one row per training-time strategy, grouped into Baseline / Training time strategy /
+ Latent Space Alignment, one column per external test chip plus a Macro Avg
(mean $\pm$ std) column, bold = best value in that column across all rows.

**Not LOFO** -- the model is trained once on all 6 `final_6_new` chips
(`--train_full_only`, no per-chip holdout) and evaluated externally against 4
*different* chips (`final_4_chip_clean_nn`) that were never seen in training at all.
So this table has 4 test-chip columns, not 6, and says "External Test Chip" instead of
"Held-Out Chip" -- there's nothing held out here, it's genuine cross-dataset transfer.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import joblib

try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(os.path.dirname(_nb)))
except Exception:
    pass

%load_ext autoreload
%autoreload 2
import config

print("CWD:", os.getcwd())


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0

CWD: /vol/bitbucket/gk225/POC_DDM/gk_code/main


In [2]:
BATCH_SIZE=128

## Config -- mirrors `slurm_jobs/f_new6_to_4chip.sh` exactly

In [3]:
EXP_FOLDER = config.FINAL_EXP_FOLDER  # plain POC_DDM_final, not nc_subtract -- matches the training job
TRAIN_GROUP = "final_6_new"           # what the model was trained on -- fixes the predictions_* filename prefix
CURVE_TYPE = "ori_curve_sg_p4_norm"

TEST_CHIPS = [
    "D20260806_E00_C00_F4500KHz_U_DDM_01_06",
    "D20260807_E00_C00_F4500KHz_U_DDM_02_07",
    "D20260808_E00_C00_F4500KHz_U_DDM_03_01",
    "D20260810_E00_C00_F4500KHz_U_DDM_04_01",
]
CHIP_LABELS = {
    "D20260806_E00_C00_F4500KHz_U_DDM_01_06": "Chip 01",
    "D20260807_E00_C00_F4500KHz_U_DDM_02_07": "Chip 02",
    "D20260808_E00_C00_F4500KHz_U_DDM_03_01": "Chip 03",
    "D20260810_E00_C00_F4500KHz_U_DDM_04_01": "Chip 04",
}

# (model_key, display_name) -- mirrors slurm_jobs/f_new6_to_4chip.sh's --array=0-4 exactly.
# cnn_gru_dual_attn_recon is BOTH the Baseline row and the anchor for the
# "+ Latent Space Alignment" section's first row (same model, pc_recenter on vs off).
MODELS = [
    ("cnn_gru_dual_attn_recon",         "CNN BiGRU + Spatial Attn"),
    ("cnn_gru_dual_attn_recon_dann",    "+ DANN"),
    ("cnn_gru_dual_attn_recon_supcon3", "+ SupCon"),
    ("cnn_gru_dual_attn_recon_aug",     "+ Temporal Aug"),
    ("cnn_gru_dual_attn_recon_mtl",     "+ MTL"),
]

## Load predictions + compute accuracy

`ground_truth()` is the exact pattern `notebooks/20260817-cross_dataset_confusion_matrices.ipynb`
already uses: map each pixel's raw well index to a label via `config.LABEL_MAPPINGS[chip_name]`,
then reconcile against the model's own `class_names` vocabulary (handles e.g. `NC` vs
`NC-ALL` naming variants).

In [4]:
def ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


def load_accuracy(chip_name, model_key, pc_recenter):
    tag = "_pc_recenter" if pc_recenter else ""
    path = (Path(EXP_FOLDER) / chip_name
            / f"predictions_{TRAIN_GROUP}_{CURVE_TYPE}_{model_key}{tag}.joblib")
    if not path.exists():
        return None, path
    d = joblib.load(path)
    y_true = ground_truth(chip_name, d["Y_well_raw"], d["class_names"])
    y_pred = np.asarray(d["pred_labels"])
    acc = float(np.mean(y_true == y_pred)) * 100
    return acc, path


# acc[model_key][pc_recenter][chip_name] = accuracy or None
acc = {m: {False: {}, True: {}} for m, _ in MODELS}
missing = []
for model_key, _ in MODELS:
    for pc_recenter in (False, True):
        for chip_name in TEST_CHIPS:
            a, path = load_accuracy(chip_name, model_key, pc_recenter)
            acc[model_key][pc_recenter][chip_name] = a
            if a is None:
                missing.append(path)

total = len(MODELS) * 2 * len(TEST_CHIPS)
if missing:
    print(f"[!] {len(missing)} / {total} prediction files not found yet -- "
          f"job/predictions still pending. Missing cells will show as \'--\' in the table below.")
    for p in missing[:10]:
        print("   ", p)
    if len(missing) > 10:
        print(f"    ... and {len(missing) - 10} more")
else:
    print("All prediction files found.")

All prediction files found.


## Build the LaTeX table

Bold = best value in that column across ALL 10 rows (both sections combined) --
matches the target format, where e.g. a chip's best score can come from the
"+ Latent Space Alignment" section rather than the baseline.

In [5]:
def fmt_cell(v, is_best):
    if v is None:
        return "--"
    s = f"{v:.2f}"
    return rf"\textbf{{{s}}}" if is_best else s


def macro_avg(values):
    vals = [v for v in values if v is not None]
    if not vals:
        return None, None
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    return float(np.mean(vals)), std


def build_table(caption, label):
    # (section_header_or_None, display_name, model_key, pc_recenter, use_quad).
    # use_quad marks "+variant" rows -- independent of whether a section header
    # prints above this row (the baseline/re-stated-baseline rows never get
    # \quad even though they open a section; "Training time strategy"'s rows
    # all get \quad even though the first one also opens a section).
    rows = [("Baseline", MODELS[0][1], MODELS[0][0], False, False)]
    rows += [("Training time strategy" if i == 0 else None, disp, key, False, True)
             for i, (key, disp) in enumerate(MODELS[1:])]
    rows += [("+ Latent Space Alignment", MODELS[0][1], MODELS[0][0], True, False)]
    rows += [(None, disp, key, True, True) for key, disp in MODELS[1:]]

    col_values = {c: [] for c in TEST_CHIPS}
    macro_values = []
    for _, _, model_key, pc_recenter, _ in rows:
        for c in TEST_CHIPS:
            col_values[c].append(acc[model_key][pc_recenter][c])
        mean, _std = macro_avg([acc[model_key][pc_recenter][c] for c in TEST_CHIPS])
        macro_values.append(mean)

    col_best = {c: max([x for x in v if x is not None], default=None) for c, v in col_values.items()}
    macro_best = max([m for m in macro_values if m is not None], default=None)

    n_chips = len(TEST_CHIPS)
    col_spec = "l " + "r" * n_chips + " r"
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \footnotesize")
    lines.append(r"    \setlength{\tabcolsep}{1.5pt}")
    lines.append(rf"    \begin{{tabular}}{{@{{}}{col_spec}@{{}}}}")
    lines.append(r"    \toprule")
    lines.append(rf"    & \multicolumn{{{n_chips}}}{{c}}{{\textbf{{External Test Chip}}}} & \\")
    lines.append(rf"    \cmidrule(lr){{2-{n_chips + 1}}}")
    header = " & ".join(rf"\textbf{{{CHIP_LABELS[c]}}}" for c in TEST_CHIPS)
    lines.append(rf"    \textbf{{Model}} & {header} & \textbf{{Macro Avg}} \\")
    lines.append(r"    \midrule")

    prev_section = None
    for section, display_name, model_key, pc_recenter, use_quad in rows:
        if section is not None:
            if prev_section is not None:
                lines.append(r"    \addlinespace")
            lines.append(rf"    \multicolumn{{{n_chips + 1}}}{{l}}{{\textbf{{{section}}}}} \\")
            prev_section = section
        row_name = rf"\quad {display_name}" if use_quad else display_name

        row_cells = []
        for c in TEST_CHIPS:
            v = acc[model_key][pc_recenter][c]
            is_best = col_best[c] is not None and v == col_best[c]
            row_cells.append(fmt_cell(v, is_best))
        cells_str = " & ".join(row_cells)

        mean, std = macro_avg([acc[model_key][pc_recenter][c] for c in TEST_CHIPS])
        if mean is None:
            macro_str = "--"
        else:
            is_best = macro_best is not None and mean == macro_best
            s = rf"{mean:.2f} $\pm$ {std:.2f}"
            macro_str = rf"\textbf{{{s}}}" if is_best else s

        lines.append(rf"    {row_name} & {cells_str} & {macro_str} \\")

    lines.append(r"    \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


N = len(TEST_CHIPS)
caption = (
    rf"Accuracy on {N} external test chips (never used in training) for the baseline, "
    r"each training-time strategy, and the same conditions with the inference-time "
    r"PC-anchored latent-space alignment additionally applied. The model in every row "
    r"is trained once on all 6 chips of the training group, with no held-out chip. "
    r"The Macro Avg column reports the mean $\pm$ standard deviation across the "
    rf"{N} test chips, and all values are percentages. Bold marks the best result in "
    r"each column across all listed conditions."
)
latex = build_table(caption, "tab:f_new6_to_4chip_result")
print(latex)

\begin{table}[htbp]
    \centering
    \caption{Accuracy on 4 external test chips (never used in training) for the baseline, each training-time strategy, and the same conditions with the inference-time PC-anchored latent-space alignment additionally applied. The model in every row is trained once on all 6 chips of the training group, with no held-out chip. The Macro Avg column reports the mean $\pm$ standard deviation across the 4 test chips, and all values are percentages. Bold marks the best result in each column across all listed conditions.}
    \label{tab:f_new6_to_4chip_result}
    \footnotesize
    \setlength{\tabcolsep}{1.5pt}
    \begin{tabular}{@{}l rrrr r@{}}
    \toprule
    & \multicolumn{4}{c}{\textbf{External Test Chip}} & \\
    \cmidrule(lr){2-5}
    \textbf{Model} & \textbf{Chip 01} & \textbf{Chip 02} & \textbf{Chip 03} & \textbf{Chip 04} & \textbf{Macro Avg} \\
    \midrule
    \multicolumn{5}{l}{\textbf{Baseline}} \\
    CNN BiGRU + Spatial Attn & 25.73 & 41.15 & 25

## Save to file

In [6]:
# out_path = Path(EXP_FOLDER) / "cross_dataset_cv" / TRAIN_GROUP / "f_new6_to_4chip_table.tex"
# out_path.parent.mkdir(parents=True, exist_ok=True)
# out_path.write_text(latex)
# print(f"[SAVED] {out_path}")

## LOFO result -- f_lf_g13_acqstart (acquisition_start alignment)

Same table format as above, but for `slurm_jobs/f_lf_g13_acqstart.sh`'s leave-one-chip-out
(LOFO) training run on `final_6_new`, using `--curve_alignment acquisition_start` instead
of `pc_ttp` (and only tasks 0-6 -- no CORAL, DANN(Conc), or `cnn_trans_dual_attn_recon`).
Columns are the 6 held-out chips; the "+ Latent Space Alignment" section applies
`08_cross_dataset_predict_new_chip.py`'s PC-anchored inference-time recentering on top of
these acquisition_start-aligned models.

In [7]:
import re
import importlib
import gc
from sklearn.metrics import confusion_matrix
import tensorflow as tf

sys.path.insert(0, 'utils')
sys.path.insert(0, 'utils/model_training')

p08   = importlib.import_module("08_cross_dataset_predict_new_chip")
vis07 = importlib.import_module("07_attribution_vis_all")
rio   = importlib.import_module("cross_dataset_result_io")
b06   = importlib.import_module("06b_cross_dataset_prediction_report")
cdt   = importlib.import_module("04_cross_dataset_training")


def short_name(folder):
    m = re.search(r'DDM_0(\d)', folder)
    return f'Chip 0{m.group(1)}' if m else folder.split('_U_', 1)[1]


def chip_sort_key(folder_or_label):
    m = re.search(r'DDM_0(\d)', folder_or_label)
    return int(m.group(1)) if m else folder_or_label


LOFO_EXP_FOLDER      = config.FINAL_EXP_FOLDER
LOFO_GROUP_NAME      = "final_6_new"
LOFO_CURVE_TYPE      = "ori_curve_sg_p4_norm"
LOFO_CURVE_ALIGNMENT = "acquisition_start"   # matches f_lf_g13_acqstart.sh's ALIGN_ARGS
LOFO_PC_TTP_ANCHOR   = "min"                 # unused for acquisition_start, kept for signature parity
LOFO_FILTER          = "noamp_remove"
LOFO_FRAC            = 0.5
LOFO_MODE_STR        = "lofo"

LOFO_CHIPS = sorted(config.CROSS_DATASET_GROUPS[LOFO_GROUP_NAME], key=chip_sort_key)
LOFO_CHIP_LABELS = {c: short_name(c) for c in LOFO_CHIPS}
lofo_exp_paths = [Path(LOFO_EXP_FOLDER, name) for name in LOFO_CHIPS]

# Same 5 models as MODELS above, but with the hyphenated "CNN-BiGRU" display name to
# match the target LaTeX format exactly.
LOFO_MODELS = [
    ("cnn_gru_dual_attn_recon",         "CNN-BiGRU + Spatial Attn"),
    ("cnn_gru_dual_attn_recon_dann",    "+ DANN"),
    ("cnn_gru_dual_attn_recon_supcon3", "+ SupCon"),
    ("cnn_gru_dual_attn_recon_aug",     "+ Temporal Aug"),
    ("cnn_gru_dual_attn_recon_mtl",     "+ MTL"),
]

In [8]:
def lofo_group_dir(group_name):
    return Path(LOFO_EXP_FOLDER) / "cross_dataset_cv" / group_name


def lofo_alignment_dir(group_name):
    return b06.alignment_dir(lofo_group_dir(group_name), LOFO_CURVE_ALIGNMENT, LOFO_PC_TTP_ANCHOR)


def load_lofo_results(group_name, curve_type, train_center_frac=LOFO_FRAC):
    out_dir = lofo_alignment_dir(group_name)
    legacy_path = b06.find_results_path(out_dir, LOFO_MODE_STR, curve_type)
    return b06.load_partitioned(out_dir, LOFO_MODE_STR, curve_type, legacy_path=legacy_path,
                                train_center_frac=train_center_frac)


lofo_results = load_lofo_results(LOFO_GROUP_NAME, LOFO_CURVE_TYPE)
_n_folds = sum(1 for k in lofo_results if k.startswith("lofo_"))
print(f"Group '{LOFO_GROUP_NAME}' ({LOFO_CURVE_ALIGNMENT}) -> {len(LOFO_CHIPS)} chips, "
      f"{_n_folds} LOFO fold(s) found on disk so far.")
for c in LOFO_CHIPS:
    tag = "" if f"lofo_{c}" in lofo_results else "  [NO CACHED RESULTS YET]"
    print(f"  - {c} ({short_name(c)}){tag}")

Group 'final_6_new' (acquisition_start) -> 6 chips, 6 LOFO fold(s) found on disk so far.
  - D20260827_E00_C00_F4500KHz_U_DDM_01_final_final (Chip 01)
  - D20260827_E00_C00_F4500KHz_U_DDM_02_final_final (Chip 02)
  - D20260827_E00_C00_F4500KHz_U_DDM_03_final_final (Chip 03)
  - D20260827_E00_C00_F4500KHz_U_DDM_04_final_final (Chip 04)
  - D20260825_E00_C00_F4500KHz_U_DDM_05_01 (Chip 05)
  - D20260825_E00_C00_F4500KHz_U_DDM_06_02 (Chip 06)


In [9]:
PC_RECENTER_CACHE_NAME = "pc_recenter_sweep_cache.joblib"

def _pc_cache_path(group_name):
    return lofo_group_dir(group_name) / PC_RECENTER_CACHE_NAME


def load_pc_cache(group_name):
    path = _pc_cache_path(group_name)
    if not path.exists():
        return {}
    try:
        return joblib.load(path)
    except Exception:
        return {}


def save_pc_cache(group_name, cache):
    path = _pc_cache_path(group_name)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(cache, path, compress=3)


def _mtime(path):
    try:
        return path.stat().st_mtime
    except FileNotFoundError:
        return None


def pc_cache_signature(model_path, out_dir, curve_type, held_out_chip=None):
    align_dir = config.cross_dataset_alignment_dir(out_dir, held_out_chip)
    return (_mtime(model_path),
           _mtime(align_dir / config.CROSS_DATASET_RESAMPLER_PATH.format(curve_type=curve_type)),
           _mtime(align_dir / config.CROSS_DATASET_PC_TTP_RECIPE_PATH.format(curve_type=curve_type)))


def lofo_ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


def lofo_pc_recenter_accuracy(group_name, curve_type, base_model, chip_path, exp_paths_all, class_names, cache,
                              train_center_frac=LOFO_FRAC):
    """Ported from RQ3_02_loco_results.ipynb / RQ3_04's combined-grid section -- computes
    (and caches, shared cache file keyed by group_name) the PC-recentered accuracy for one
    base model on one held-out chip, via p08.predict_new_chip(..., pc_recenter=True, ...)."""
    out_dir = lofo_group_dir(group_name)
    chip_name = chip_path.name
    fold_label = f"lofo_{chip_name}"
    model_dir = out_dir / "model_interpretation" / fold_label
    model_path = model_dir / f"{base_model}_{LOFO_FILTER}_{curve_type}{rio.frac_suffix(train_center_frac)}_model.keras"
    if not model_path.exists():
        return None

    cache_key = (curve_type, LOFO_CURVE_ALIGNMENT, LOFO_PC_TTP_ANCHOR, train_center_frac, LOFO_FILTER,
                base_model, chip_name)
    sig = pc_cache_signature(model_path, out_dir, curve_type, held_out_chip=chip_name)
    cached = cache.get(cache_key)
    if cached is not None and cached["sig"] == sig and "cm_base" in cached["row"]:
        return cached["row"]

    align_result = p08.align_new_chip(chip_path, out_dir, curve_type, LOFO_CURVE_ALIGNMENT, LOFO_PC_TTP_ANCHOR,
                                      group_name=group_name, held_out_chip=chip_name)
    if align_result is None:
        return None
    curves, resampler, Y_well_raw, pc_curves_aligned, coords, well_ids = align_result

    loaded = vis07.load_saved_models(model_dir, LOFO_FILTER, len(resampler.t_grid), curve_type=curve_type,
                                     model_names=[base_model], train_center_frac=train_center_frac)
    if base_model not in loaded:
        return None
    model = loaded[base_model]
    if p08._is_spatial(base_model) and (coords is None or well_ids is None):
        return None

    y_true = lofo_ground_truth(chip_name, Y_well_raw, class_names)
    valid = y_true != "PC"
    exp_paths_train = [p for p in exp_paths_all if p.name != chip_name]

    probs_base, _ = p08.predict_new_chip(model, base_model, curves, coords, well_ids, pc_curves_aligned,
                                         exp_paths_train, out_dir, curve_type, LOFO_FILTER, LOFO_CURVE_ALIGNMENT,
                                         pc_recenter=False, held_out_chip=chip_name, batch_size=BATCH_SIZE)
    pred_base = np.array(class_names)[np.argmax(probs_base, axis=1)]
    acc_base = (pred_base[valid] == y_true[valid]).mean() if valid.any() else float('nan')

    acc_recenter = float('nan')
    try:
        probs_r, _ = p08.predict_new_chip(model, base_model, curves, coords, well_ids, pc_curves_aligned,
                                          exp_paths_train, out_dir, curve_type, LOFO_FILTER, LOFO_CURVE_ALIGNMENT,
                                          pc_recenter=True, force_rerun=True, held_out_chip=chip_name, batch_size=BATCH_SIZE)
        pred_r = np.array(class_names)[np.argmax(probs_r, axis=1)]
        acc_recenter = (pred_r[valid] == y_true[valid]).mean() if valid.any() else float('nan')
    except ValueError:
        pass   # no PC snapshot for this chip -- leave acc_recenter as NaN

    row = {"acc_baseline": acc_base * 100, "acc_pc_recenter": acc_recenter * 100, "n_pixels": int(valid.sum()),
          "cm_base": True, "class_names": list(class_names)}
    cache[cache_key] = {"sig": sig, "row": row}
    save_pc_cache(group_name, cache)

    del model, loaded
    tf.keras.backend.clear_session()
    gc.collect()
    return row


_lofo_pc_cache = load_pc_cache(LOFO_GROUP_NAME)

In [10]:
def lofo_base_accuracy(chip_name, model_key, filter_name=LOFO_FILTER):
    preds_key = config.MODEL_KEY_MAP.get(model_key, (None,))[0]
    if preds_key is None:
        return None
    fold_entry = lofo_results.get(f"lofo_{chip_name}", {})
    res_entry = fold_entry.get(filter_name)
    if res_entry is None or preds_key not in res_entry:
        return None
    y_true = np.concatenate(res_entry["y_trues_"])
    y_pred = np.concatenate(res_entry[preds_key])
    return float(np.mean(y_true == y_pred)) * 100


def lofo_recenter_accuracy(chip_name, model_key):
    class_names = lofo_results.get(f"lofo_{chip_name}", {}).get("class_names")
    if class_names is None:
        return None
    chip_path = Path(LOFO_EXP_FOLDER, chip_name)
    result = lofo_pc_recenter_accuracy(LOFO_GROUP_NAME, LOFO_CURVE_TYPE, model_key, chip_path,
                                       lofo_exp_paths, class_names, _lofo_pc_cache,
                                       train_center_frac=LOFO_FRAC)
    if result is None:
        return None
    v = result.get("acc_pc_recenter")
    return v if v is not None and not np.isnan(v) else None


# lofo_acc[model_key][pc_recenter][chip_name] = accuracy or None
lofo_acc = {m: {False: {}, True: {}} for m, _ in LOFO_MODELS}
lofo_missing = []
for model_key, _ in LOFO_MODELS:
    for chip_name in LOFO_CHIPS:
        a = lofo_base_accuracy(chip_name, model_key)
        lofo_acc[model_key][False][chip_name] = a
        if a is None:
            lofo_missing.append(("base", model_key, chip_name))

        ar = lofo_recenter_accuracy(chip_name, model_key)
        lofo_acc[model_key][True][chip_name] = ar
        if ar is None:
            lofo_missing.append(("pc_recenter", model_key, chip_name))

_lofo_total = len(LOFO_MODELS) * 2 * len(LOFO_CHIPS)
if lofo_missing:
    print(f"[!] {len(lofo_missing)} / {_lofo_total} (model, condition, chip) combos not available yet "
         f"-- job still training / recenter cache still filling. Missing cells will show as '--'.")
    for tag, model_key, chip_name in lofo_missing[:10]:
        print(f"    {tag:12s} {model_key:32s} {short_name(chip_name)}")
    if len(lofo_missing) > 10:
        print(f"    ... and {len(lofo_missing) - 10} more")
else:
    print("All LOFO base + pc_recenter results available.")

  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 1686 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260827_E00_C00_F4500KHz_U_DDM_01_final_final


/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 86 variables whereas the saved optimizer has 82 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 82 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
I0000 00:00:1788408010.157489 1889290 service.cc:145] XLA service 0x3e23c650 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788408010.157520 1889290 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4080, Compute Capability 8.9
I0000 00:00:1788408010.211431 1889290 device_compiler.h:188] Compiled cluster using XLA!  This li

  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/cross_dataset_cv/final_6_new/model_interpretation/lofo_D20260827_E00_C00_F4500KHz_U_DDM_01_final_final/pc_reference_embedding_cnn_gru_dual_attn_recon_supcon3_noamp_remove_ori_curve_sg_p4_norm.joblib
  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 1901 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260825_E00_C00_F4500KHz_U_DDM_05_01
  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/cross_dataset_cv/final_6_new/model_interpretation/lofo_D20260825_E00_C00_F4500KHz_U_DDM_05_01/pc_reference_embedding_cnn_gru_dual_attn_recon_supcon3_noamp_remove_ori_curve_sg_p4_norm.joblib
  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 3637 samples from wells [6, 8, 9] (y_label={6: 'IBV', 8: 'PC', 9: 'NC-ALL'}, y_concentration={6: '1e+04', 8: '0e+00', 9: '0e+00'}) for D20260825_E00_C00_F4500KHz_U_DDM_

In [11]:
def build_lofo_table(caption, label):
    rows = [("Baseline", LOFO_MODELS[0][1], LOFO_MODELS[0][0], False, False)]
    rows += [("Training time strategy" if i == 0 else None, disp, key, False, True)
             for i, (key, disp) in enumerate(LOFO_MODELS[1:])]
    rows += [("+ Latent Space Alignment", LOFO_MODELS[0][1], LOFO_MODELS[0][0], True, False)]
    rows += [(None, disp, key, True, True) for key, disp in LOFO_MODELS[1:]]

    col_values = {c: [] for c in LOFO_CHIPS}
    macro_values = []
    for _, _, model_key, pc_recenter, _ in rows:
        for c in LOFO_CHIPS:
            col_values[c].append(lofo_acc[model_key][pc_recenter][c])
        mean, _std = macro_avg([lofo_acc[model_key][pc_recenter][c] for c in LOFO_CHIPS])
        macro_values.append(mean)

    col_best = {c: max([x for x in v if x is not None], default=None) for c, v in col_values.items()}
    macro_best = max([m for m in macro_values if m is not None], default=None)

    n_chips = len(LOFO_CHIPS)
    col_spec = "l " + "r" * n_chips + " r"
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \footnotesize")
    lines.append(r"    \setlength{\tabcolsep}{1.5pt}")
    lines.append(rf"    \begin{{tabular}}{{@{{}}{col_spec}@{{}}}}")
    lines.append(r"    \toprule")
    lines.append(rf"    & \multicolumn{{{n_chips}}}{{c}}{{\textbf{{Held-Out Chip}}}} & \\")
    lines.append(rf"    \cmidrule(lr){{2-{n_chips + 1}}}")
    header = " & ".join(rf"\textbf{{{LOFO_CHIP_LABELS[c]}}}" for c in LOFO_CHIPS)
    lines.append(rf"    \textbf{{Model}} & {header} & \textbf{{Macro Avg}} \\")
    lines.append(r"    \midrule")

    prev_section = None
    for section, display_name, model_key, pc_recenter, use_quad in rows:
        if section is not None:
            if prev_section is not None:
                lines.append(r"    \addlinespace")
            lines.append(rf"    \multicolumn{{{n_chips + 1}}}{{l}}{{\textbf{{{section}}}}} \\")
            prev_section = section
        row_name = rf"\quad {display_name}" if use_quad else display_name

        row_cells = []
        for c in LOFO_CHIPS:
            v = lofo_acc[model_key][pc_recenter][c]
            is_best = col_best[c] is not None and v == col_best[c]
            row_cells.append(fmt_cell(v, is_best))
        cells_str = " & ".join(row_cells)

        mean, std = macro_avg([lofo_acc[model_key][pc_recenter][c] for c in LOFO_CHIPS])
        if mean is None:
            macro_str = "--"
        else:
            is_best = macro_best is not None and mean == macro_best
            s = rf"{mean:.2f} $\pm$ {std:.2f}"
            macro_str = rf"\textbf{{{s}}}" if is_best else s

        lines.append(rf"    {row_name} & {cells_str} & {macro_str} \\")

    lines.append(r"    \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


_lofo_caption = (
    r"Leave-one-chip-out accuracy for each held-out chip, for the baseline, each "
    r"training time strategy, and the same conditions with the inference time "
    r"control anchored latent space alignment additionally applied (curve alignment: "
    r"acquisition\_start). The Macro Avg column reports the mean $\pm$ standard "
    r"deviation across the six held-out chips, and all values are percentages. Bold "
    r"marks the best result in each held-out chip across all listed conditions."
)
lofo_latex = build_lofo_table(_lofo_caption, "tab:lofo_generalisability_result_acqstart")
print(lofo_latex)

\begin{table}[htbp]
    \centering
    \caption{Leave-one-chip-out accuracy for each held-out chip, for the baseline, each training time strategy, and the same conditions with the inference time control anchored latent space alignment additionally applied (curve alignment: acquisition\_start). The Macro Avg column reports the mean $\pm$ standard deviation across the six held-out chips, and all values are percentages. Bold marks the best result in each held-out chip across all listed conditions.}
    \label{tab:lofo_generalisability_result_acqstart}
    \footnotesize
    \setlength{\tabcolsep}{1.5pt}
    \begin{tabular}{@{}l rrrrrr r@{}}
    \toprule
    & \multicolumn{6}{c}{\textbf{Held-Out Chip}} & \\
    \cmidrule(lr){2-7}
    \textbf{Model} & \textbf{Chip 01} & \textbf{Chip 02} & \textbf{Chip 03} & \textbf{Chip 04} & \textbf{Chip 05} & \textbf{Chip 06} & \textbf{Macro Avg} \\
    \midrule
    \multicolumn{7}{l}{\textbf{Baseline}} \\
    CNN-BiGRU + Spatial Attn & \textbf{66.57} & 43